# IC-1102 — Case study: which cutoff?

**ML & FinTech · 115-1 · 20261102 · Session 3 · work in pairs · 40 minutes, then discussion**

| Step | Min | What you do |
|---|---|---|
| 0 | 3 | Sit with a partner. Fill in both IDs; your desk has its own applicants **and its own cost of a missed default** |
| 1 | 10 | A confusion matrix **by hand**: accuracy, sensitivity, specificity |
| 2 | 12 | Your desk: the model versus "approve everyone" |
| 3 | 10 | Pick a cutoff for your desk's costs; compare with your partner |
| 4 | 5 | Hand in **one sheet of paper per pair** |

You may use any AI tool for the code. The graded part is the paper your pair hands in, and what
you say when called on. Your numbers come from your 學號; your partner's differ.

---

## The case

The bank's dashboard reports the default model is **82% accurate**. Your manager says:

> *"We also found that simply **approving everyone** is 92% accurate. Almost the same, so no model needed.
> And if we do keep the model, the cutoff is 0.5 — that is what 'probability of default' means."*

Each desk is different: on **your** desk a missed default costs **c** times as much as a refused
good customer (your c is printed in Step 0). Your manager asks: **"Model or no model? And if the
model, which cutoff?"**

## Setup

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

In [ ]:
STUDENT_ID = ""    # your 學號, in quotes, e.g. "315707999"
PARTNER_ID = ""    # your partner's 學號

In [ ]:
URL  = ("https://raw.githubusercontent.com/202609-ML-FinTech/00-course-info/main/slides/"
        "default%20of%20credit%20card%20clients.csv")
CAND = ["default of credit card clients.csv", "../slides/default of credit card clients.csv",
        "../00-course-info/slides/default of credit card clients.csv"]
path = next((p for p in CAND if Path(p).exists()), URL)
credit = pd.read_csv(path).rename(columns={"X1":"LIMIT","X5":"AGE","X6":"PAY_0","X7":"PAY_2",
                                           "X12":"BILL_1","X18":"PAY_AMT_1","Y":"default"})
credit["LIMIT"] = credit["LIMIT"] / 10000          # NT$ ten-thousands
credit["BILL_1"] = credit["BILL_1"] / 10000
credit["PAY_AMT_1"] = credit["PAY_AMT_1"] / 10000
credit = credit[credit["PAY_0"] <= 8]                # one record has PAY_0 = 3032: a data error (IC-02). Dropped.
# 30,000 Taiwanese credit-card clients (2005). default = 1 if the client defaulted next month.
# PAY_0: months late on the most recent payment (-1 = paid on time, 0 = revolving, 1..9 months late).

def my_sample(student_id, n=3000):
    """Your desk's 3,000 applicants. The same 學號 always gives the same applicants."""
    seed = int(hashlib.sha256(f"{int(student_id)}-credit".encode()).hexdigest(), 16) % 2**32
    rng = np.random.default_rng(seed)
    return credit.iloc[np.sort(rng.choice(len(credit), n, replace=False))].copy()

assert str(STUDENT_ID).strip().isdigit(), "Fill in STUDENT_ID in the cell above, then run again."
desk = my_sample(STUDENT_ID)
print(f"Desk of {STUDENT_ID}: {len(desk)} applicants, {desk['default'].mean()*100:.1f}% defaulted")
MY_COST = 3 + int(hashlib.sha256(f"{int(STUDENT_ID)}-cost".encode()).hexdigest(), 16) % 6
print(f"On my desk a missed default costs {MY_COST} times a refused good customer.")

---
# Step 1 — One confusion matrix by hand (10 min)

The model, applied to 1,000 applicants at cutoff 0.5:

| | predicted default | predicted no default |
|---|---|---|
| **truly defaulted** | 56 | 165 |
| **truly paid** | 20 | 759 |

**Q1.** By hand: accuracy = ____ , sensitivity (defaulters caught) = ____ , specificity (good customers cleared) = ____ .

**Q2.** For the same 1,000 applicants, "approve everyone" has accuracy ____ , sensitivity ____ , specificity ____ .
Which of the two would you rather be the bank? One sentence.

> _replace this line_

---
# Step 2 — Your desk (12 min)

Split (given), fit a logistic regression of `default` on `LIMIT, PAY_0, PAY_2, BILL_1, PAY_AMT_1`
on `train`, and get `p = predict_proba(test)[:, 1]`. Then print, on `test`:
accuracy of "approve everyone", and accuracy, sensitivity, specificity of the model at cutoff 0.5.

In [ ]:
from sklearn.model_selection import train_test_split
seed = int(hashlib.sha256(f"{int(STUDENT_ID)}-split".encode()).hexdigest(), 16) % 2**32
train, test = train_test_split(desk, test_size=0.4, random_state=seed, stratify=desk["default"])
print(len(train), "to fit,", len(test), "to test")

In [ ]:
# 2: fit; p on test; the two accuracies, sensitivity and specificity at 0.5



**Q3.** My desk: approve-everyone accuracy ____ ; model at 0.5: accuracy ____ , sensitivity ____ , specificity ____ .

**Q4.** Is the manager right that the two are "the same"? What does accuracy hide here?

> _replace this line_

---
# Step 3 — The cutoff for your desk (10 min)

For cutoffs 0.1, 0.2, 0.3, 0.4, 0.5 compute on `test`:
expected cost = (false positives × 1) + (false negatives × MY_COST). Print the table and pick the cheapest.

In [ ]:
# 3: cost at each cutoff



**Q5.** My c = ____ ; cheapest cutoff = ____ ; at that cutoff sensitivity = ____ and specificity = ____ .

**Q6.** Swap with your partner.

| | c | best cutoff | sensitivity there |
|---|---|---|---|
| me | | | |
| my partner | | | |

Same model, different cutoffs — why? Is 0.5 special at all?

> _replace this line_

---
# Step 4 — Hand in, on paper (5 min)

**One sheet per pair.** Write:

1. Both names and both 學號.
2. For **each** of you: approve-everyone accuracy, the model's sensitivity at 0.5, your c and your chosen cutoff.
3. **Two sentences, agreed between you**, answering the client's question and the trap.

Give the sheet to Prof. Teng or Claire before you leave. **The sheet is the graded part.**

Then: *Restart & Run All*, and push this notebook as `in-class-exercise/1102/IC-1102.ipynb`
in your own repository before **12:10**.